# MQR-v1: two-seed initialization learning pilot

Attach the verified tuning dataset and completed batch-tuning notebook outputs. Uses the selected common microbatch, two independent T4 processes, seven fresh arms per seed, fixed validation banks and the registered short pilot schedule. Only initialization variants change; canonical architecture/defaults remain intact. Parallel run timings support no speed claim. Checkpoints are retained selectively; the evidence ZIP excludes their large payloads. See docs/TUNING_PROTOCOL.md.


In [ ]:
from pathlib import Path
import importlib.metadata
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

# Normally autodetected. Set a mounted directory or ZIP path only if ambiguous.
source_roots = [p.parent for p in Path("/kaggle/input").rglob("package_manifest.json")
                if "mqr-v1-tuning" in p.parts and (p.parent / "src/mqr").is_dir()]
assert len(source_roots) == 1, f"Expected one original tuning source/data upload: {source_roots}"
UPLOAD_ROOT = source_roots[0]
CODE_ROOT = Path("/kaggle/working/mqr-source")
assert sys.version_info >= (3, 11), "Python 3.11+ is required"
if UPLOAD_ROOT is None:
    mounted = Path("/kaggle/input")
    candidates = list(mounted.rglob("mqr-kaggle-readiness.zip"))
    candidates += [marker.parent for marker in mounted.rglob("pyproject.toml")
                   if (marker.parent / "src" / "mqr").is_dir()]
    if len(candidates) != 1:
        raise ValueError("Set UPLOAD_ROOT to the intended source directory or ZIP; found: "
                         + repr([str(path) for path in candidates]))
    UPLOAD_ROOT = candidates[0]
else:
    UPLOAD_ROOT = Path(UPLOAD_ROOT)
assert UPLOAD_ROOT.exists(), f"Source upload does not exist: {UPLOAD_ROOT}"
print("Selected upload:", UPLOAD_ROOT)
if not CODE_ROOT.exists():
    archives = [UPLOAD_ROOT] if UPLOAD_ROOT.is_file() else list(UPLOAD_ROOT.glob("*.zip"))
    if archives:
        assert len(archives) == 1, "Choose the exact source ZIP using UPLOAD_ROOT"
        CODE_ROOT.mkdir(parents=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                target = (CODE_ROOT / item.filename).resolve()
                assert target.is_relative_to(CODE_ROOT.resolve()), "ZIP path escapes source directory"
                assert (item.external_attr >> 16) & 0o170000 != 0o120000, "ZIP symlinks are not supported"
            archive.extractall(CODE_ROOT)
    else:
        shutil.copytree(UPLOAD_ROOT, CODE_ROOT)
else:
    print("Reusing copied source; choose a fresh CODE_ROOT when changing the uploaded ZIP")
projects = list(CODE_ROOT.rglob("pyproject.toml"))
assert len(projects) == 1, "Source upload must contain one project pyproject.toml"
PROJECT = projects[0].parent
package_manifest = PROJECT / "package_manifest.json"
if package_manifest.exists():
    payload = json.loads(package_manifest.read_text(encoding="utf-8"))
    for name, expected in payload["files"].items():
        artifact = (PROJECT / name).resolve()
        assert artifact.is_relative_to(PROJECT.resolve()), "Manifest path escapes project"
        digest = hashlib.sha256()
        with artifact.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1048576), b""):
                digest.update(chunk)
        assert artifact.stat().st_size == expected["bytes"], f"Package size mismatch: {name}"
        assert digest.hexdigest() == expected["sha256"], f"Package hash mismatch: {name}"
    print("Verified packaged payload files:", len(payload["files"]))
# Use the existing CUDA torch; do not resolve or reinstall project dependencies.
assert int(importlib.metadata.version("setuptools").split(".")[0]) >= 77, "Install setuptools>=77 separately, then rerun setup"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(PROJECT)])
# Editable .pth files are processed at startup, so expose source to this live kernel.
sys.path.insert(0, str(PROJECT / "src"))
print("Installed source:", PROJECT)
print("Included prepared data:", PROJECT / "prepared")


def require_tokenizers():
    """Bootstrap only the tokenizer wheel before any tokenizers import."""
    def supported(value):
        try:
            version = tuple(map(int, value.split(".")[:2]))
        except (AttributeError, TypeError, ValueError):
            return False
        return len(version) == 2 and (0, 21) <= version < (0, 23)

    try:
        installed = importlib.metadata.version("tokenizers")
    except importlib.metadata.PackageNotFoundError:
        installed = None
    loaded = sys.modules.get("tokenizers")
    if loaded is not None and getattr(loaded, "__version__", None) != installed:
        raise RuntimeError("A different tokenizers version is already loaded; restart the clean kernel before setup")
    if not supported(installed):
        if any(name == "tokenizers" or name.startswith("tokenizers.") for name in sys.modules):
            raise RuntimeError("tokenizers is already imported; restart the clean kernel before changing its wheel")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps",
                               "--only-binary=:all:", "tokenizers==0.22.2"])
        importlib.invalidate_caches()
        installed = importlib.metadata.version("tokenizers")
        if installed != "0.22.2":
            raise RuntimeError("Tokenizer installation did not expose the requested tokenizers==0.22.2 wheel")
    if not supported(installed):
        raise RuntimeError("Prepared BPE data requires tokenizers>=0.21,<0.23")
    return installed

# Run before importing MQR or any library that could load tokenizers.
print("Tokenizers runtime:", require_tokenizers())


In [ ]:
import time
import torch
from mqr.provenance import prepared_identity, file_sha256

DATA_ROOT = PROJECT / "prepared"
identity = prepared_identity(DATA_ROOT)
candidates = []
for path in Path("/kaggle/input").rglob("summary.json"):
    value = json.loads(path.read_text(encoding="utf-8"))
    if value.get("protocol") == "fixed-global32-microbatch-sweep-v1":
        candidates.append((path, value))
assert len(candidates) == 1, "Attach exactly one completed batching run"
TUNING_PATH, tuning = candidates[0]
assert tuning["data_identity"] == identity
assert tuning["best_shared_batch"] is not None
assert tuning.get("protocol_completed"), "Complete batching/profile protocol before learning pilot"
MICROBATCH = tuning["best_shared_batch"]
assert MICROBATCH in (1, 2, 4, 8, 16, 32)
# Bind performance selection to the same implementation used by this pilot.
for job in tuning["jobs"]:
    if job["status"] == "complete":
        worker = json.loads((TUNING_PATH.parent / job["directory"] / "provenance.json").read_text())
        for name, expected in worker["source_sha256"].items():
            assert file_sha256(PROJECT / "src/mqr" / name) == expected, f"Source changed since tuning: {name}"
assert torch.cuda.device_count() >= 2, "Select T4 x2 for the two independent seed processes"
RUN_ROOT = Path("/kaggle/working") / ("mqr-learning-" + time.strftime("%Y%m%d-%H%M%S"))
RUN_ROOT.mkdir()
launch = {"protocol": "mqr-learning-pilot-v1", "seed_devices": {"0": "cuda:0", "1": "cuda:1"},
          "microbatch": MICROBATCH, "effective_batch_sequences": 32,
          "source_tuning_summary_sha256": file_sha256(TUNING_PATH),
          "data_identity": identity, "parallel_timings_used_for_speed_claims": False}
(RUN_ROOT / "launch.json").write_text(json.dumps(launch, indent=2) + "\n", encoding="utf-8")
print(json.dumps(launch, indent=2))


In [ ]:
import subprocess

workers, handles = [], []
deadline = time.monotonic() + 6600
try:
    for seed in (0, 1):
        log = (RUN_ROOT / f"seed{seed}.log").open("w", encoding="utf-8")
        handles.append(log)
        command = [sys.executable, "-m", "mqr.pilot", "--data-dir", str(DATA_ROOT),
                   "--output-root", str(RUN_ROOT / f"seed{seed}"), "--seed", str(seed),
                   "--device", f"cuda:{seed}", "--microbatch", str(MICROBATCH)]
        workers.append(subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT))
    while any(worker.poll() is None for worker in workers):
        if any(worker.poll() not in (None, 0) for worker in workers):
            raise RuntimeError("A seed process failed; inspect retained seed logs")
        if time.monotonic() >= deadline:
            raise TimeoutError("Learning pilot exceeded its bounded runtime")
        time.sleep(5)
    assert all(worker.returncode == 0 for worker in workers), "A seed process failed"
finally:
    for worker in workers:
        if worker.poll() is None:
            worker.terminate()
    for worker in workers:
        try:
            worker.wait(timeout=15)
        except subprocess.TimeoutExpired:
            worker.kill()
            worker.wait(timeout=15)
    for handle in handles:
        handle.close()


In [ ]:
reports = [json.loads((RUN_ROOT / f"seed{seed}/report.json").read_text()) for seed in (0, 1)]
assert all(report["status"] == "complete" for report in reports)
assert reports[0]["validation_bank"]["bank_sha256"] == reports[1]["validation_bank"]["bank_sha256"]
rows = []
for report in reports:
    for arm, result in report["results"].items():
        assert result["completed_steps"] == 128 and result["final_state_audit"]["finite"]
        endpoint = result["evaluations"]["128"]
        rows.append({"seed": report["seed"], "arm": arm, "step": 128,
                     "macro_validation_nll": endpoint["macro_nll"],
                     "per_language_nll": {k: v["full_nll"] for k, v in endpoint["per_language"].items()},
                     "amp_overflows": result["amp_overflows"],
                     "checkpoint": result["checkpoint"]})
(RUN_ROOT / "endpoint_summary.json").write_text(json.dumps(rows, indent=2) + "\n", encoding="utf-8")
# Downloadable small evidence archive; retained full checkpoints remain separate.
with zipfile.ZipFile(str(RUN_ROOT) + "-evidence.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and path.suffix in (".json", ".jsonl", ".log"):
            archive.write(path, path.relative_to(RUN_ROOT))
print(json.dumps(rows, indent=2))
print("Small evidence archive:", str(RUN_ROOT) + "-evidence.zip")
print("Full checkpoint files remain in each seed/arm output directory.")
